# RCC RefCert: guided first run

Explore finite-control quantum models in Reference-Contingent Complexity (RCC), a framework that makes the physical reference dependence of quantum circuit complexity explicit. This notebook studies how a declared process and its program prior compare with the reference background.

A program semidensity is the sum of terminating output states weighted by their program code probabilities; its trace records the total weight of terminating programs. A reference certificate bounds this output sum relative to the reference state.

You will compute these objects, compare local reference gain with code length, and examine a sufficient prefix re-encoding (Eq. H.76). The examples reproduce selected Appendix F/H constructions and exercise the same kernel available for new models.

Section and equation references follow manuscript version 4; the [paper map](docs/PAPER_MAP.md) explains the version alignment.

Follow the [notebook setup instructions](README.md#jupyter-notebook) and select **Python (rcc-refcert)** as the kernel. To preview the output, view the [example results](results/reference_report.md).


In [ ]:
import sys
from importlib.util import find_spec
from pathlib import Path

import numpy as np

source = Path.cwd() / "src"
if find_spec("rcc_refcert") is None and (source / "rcc_refcert").is_dir():
    sys.path.insert(0, str(source))

from rcc_refcert import audit_case, get_case
from rcc_refcert.cli import main

## 1. See the three reference cases

Each case has a distinct job: a transparent demonstration, a nontrivial certificate diagnostic, and a least-fixed-point boundary.

In [ ]:
_ = main(["examples"])

## 2. Run the transparent case

`dephase-or-halt` has a closed-form semidensity. Every supplied certificate route is expected to pass.

In [ ]:
_ = main(["example", "dephase-or-halt"])

## 3. Inspect the program semidensity

The fixed-point output is the terminating program semidensity, whose trace is the total Kraft weight of accepted programs. The current case has complete local codebooks. With incomplete codes, the trace deficit can also include unused code space; a finite partial sum additionally omits programs that halt later. The minimum reference-domination calculation (Eq. H.34) compares the full semidensity with the reference state. See the [halting conventions](docs/CONVENTIONS.md#halting-and-subnormalization).

In [ ]:
transparent = audit_case(get_case("dephase-or-halt"))
analysis = transparent.model_analysis
semidensity = analysis.fixed_point.output
if semidensity is None or analysis.fixed_model_domination is None:
    raise RuntimeError("The transparent case did not produce its fixed-model result.")
trace = float(np.trace(semidensity).real)
nonhalting_mass = max(0.0, 1.0 - trace)
print("Program semidensity M:")
print(np.real_if_close(semidensity))
print(f"trace(M): {trace:.12g}")
print(f"nonhalting mass: {nonhalting_mass:.12g}")
print(f"fixed-model C*: {analysis.fixed_model_domination.constant:.12g}")

## 4. Inspect a useful route failure

The multiblock case exercises unequal block dimensions and rectangular Kraus maps. Its original codewords deliberately fail the local gain–cost sufficient condition (Eq. H.70), while the supplied Bellman–Choi and reference-potential certificates pass their numerical checks. The sufficient prefix re-encoding (Eq. H.76) meets the local criterion for a separately re-encoded model.

In [ ]:
_ = main(["example", "multiblock-rectangular", "--details"])

### Inspect the failed local test and the proposed prefix codes

Compare the weighted gain sums for the original and proposed codewords. The local sufficient condition requires a sum no greater than one at each syntax state; the table also shows each action's reference gain.

In [ ]:
multiblock = audit_case(get_case("multiblock-rectangular"))
if multiblock.gain_cost is None:
    raise RuntimeError("The multiblock case did not produce its gain-cost report.")
for syntax in multiblock.gain_cost.syntax_reports:
    print(
        f"{syntax.syntax_state}: current weighted gain sum {syntax.current_weighted_sum:.12g} "
        f"-> proposed weighted gain sum {syntax.suggested_weighted_sum:.12g}"
    )
    for action in syntax.actions:
        print(
            f"  {action.action_name}: gain {action.reference_gain:.12g}; "
            f"code {action.current_codeword} -> {action.suggested_codeword}"
        )

## 5. Inspect the nonhalting boundary

Here the full transient spectral radius is one. The least-fixed-point semantics remains valid while the linear inverse reaches its spectral boundary, so the minimum reference-domination constant remains unevaluated by this method. A finite truncation alone does not determine it.

In [ ]:
_ = main(["example", "dark-nonhalting", "--details"])

## 6. Reproduce the complete frozen suite

The final command runs all three cases and the Appendix F witnesses, then compares the normalized suite structure and its report with the checked-in references without overwriting them.

In [ ]:
status = main(["reproduce", "--check"])
if status != 0:
    raise RuntimeError("The frozen reference suite did not match.")

## What the run establishes

These cells establish finite-input and fixed-model numerical statements for declared processes and supplied reference certificates. They make selected Appendix F/H constructions inspectable. The associated RCC paper supplies the physical reference-consistency, faithful-transcription and uniform model-family arguments.

- [Understand the evidence and scientific scope](docs/SCIENTIFIC_SCOPE.md).
- [Build a finite-control model](docs/MODEL_GUIDE.md) or follow the [custom-model workflow](docs/CUSTOM_MODELS.md).
- [Calculate terminal-state lower bounds](RCC_Bounds_Quickstart.ipynb).
- [Share and reverify an analysis](docs/ANALYSIS_BUNDLES.md).
